# GibbsReactor Tutorial — Chemical Equilibrium with NeqSim

This notebook demonstrates how to use the `GibbsReactor` and `GibbsReactorCO2` classes
for computing chemical equilibrium via Gibbs free energy minimisation.

**Examples covered:**
1. Methane combustion (adiabatic flame temperature)
2. Ammonia synthesis (high pressure, isothermal)
3. Sulfur formation from sour gas (Claus-like)
4. Acid gas equilibrium in CO₂ transport (`GibbsReactorCO2`)
5. Convergence diagnostics and Jacobian inspection

In [1]:
# Cell 1 — Setup: dual-boot devtools / pip
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except ImportError:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim\target\neqsim-3.7.0.jar

JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes
All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)


In [2]:
# Cell 2 — Import classes based on mode
import jpype

if NEQSIM_MODE == "devtools":
    SystemSrkEos = ns.SystemSrkEos
    SystemSrkCPAstatoil = ns.SystemSrkCPAstatoil
    Stream = ns.Stream
    ProcessSystem = ns.ProcessSystem
    GibbsReactor = ns.GibbsReactor
    GibbsReactorCO2 = jpype.JClass("neqsim.process.equipment.reactor.GibbsReactorCO2")
else:
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    Stream = jneqsim.process.equipment.stream.Stream
    ProcessSystem = jneqsim.process.processmodel.ProcessSystem
    GibbsReactor = jneqsim.process.equipment.reactor.GibbsReactor
    GibbsReactorCO2 = jneqsim.process.equipment.reactor.GibbsReactorCO2

print(f"Classes loaded in {NEQSIM_MODE} mode")

Classes loaded in devtools mode


---
## Example 1 — Adiabatic Methane Combustion

Burn methane with excess oxygen/nitrogen and compute the **adiabatic flame temperature**.
The reactor adjusts temperature until $H_{\text{out}} = H_{\text{in}}$.

In [3]:
# Cell 3 — Methane combustion (adiabatic)
fluid = SystemSrkEos(598.0, 100.0)
fluid.addComponent("methane", 0.25)
fluid.addComponent("oxygen", 1.0)
fluid.addComponent("nitrogen", 1.0)
fluid.addComponent("CO2", 0.0)
fluid.addComponent("water", 0.0)
fluid.addComponent("CO", 0.0)
fluid.setMixingRule(2)

feed = Stream("Combustion Feed", fluid)
feed.run()

reactor = GibbsReactor("Combustion Reactor", feed)
reactor.setEnergyMode(GibbsReactor.EnergyMode.ADIABATIC)
reactor.setDampingComposition(0.01)
reactor.setMaxIterations(5000)
reactor.setConvergenceTolerance(1e-3)
reactor.run()

outlet = reactor.getOutletStream().getThermoSystem()
T_flame = outlet.getTemperature() - 273.15

print(f"Converged: {reactor.hasConverged()}")
print(f"Iterations: {reactor.getActualIterations()}")
print(f"Adiabatic flame temperature: {T_flame:.1f} °C")
print(f"Enthalpy of reactions: {reactor.getEnthalpyOfReactions():.2f} kJ")
print(f"Power: {reactor.getPower('kW'):.2f} kW")
print()
print("Outlet composition (mole fractions):")
for i in range(outlet.getNumberOfComponents()):
    z = outlet.getComponent(i).getz()
    name = outlet.getComponent(i).getComponentName()
    if z > 1e-8:
        print(f"  {name:15s}: {z:.6f}")

Converged: True
Iterations: 100
Adiabatic flame temperature: 1990.3 °C
Enthalpy of reactions: -187.05 kJ
Power: 187.05 kW

Outlet composition (mole fractions):
  methane        : 0.000000
  oxygen         : 0.222250
  nitrogen       : 0.444426
  CO2            : 0.111029
  water          : 0.222215
  CO             : 0.000079


---
## Example 2 — Ammonia Synthesis (Isothermal, High Pressure)

The Haber-Bosch reaction: $N_2 + 3H_2 \rightleftharpoons 2NH_3$.
At high pressure (300 bar) and moderate temperature (450 K), equilibrium strongly favours ammonia.

In [4]:
# Cell 4 — Ammonia synthesis
fluid_nh3 = SystemSrkEos(298.0, 100.0)
fluid_nh3.addComponent("hydrogen", 1.5)
fluid_nh3.addComponent("nitrogen", 0.5)
fluid_nh3.addComponent("ammonia", 0.0)
fluid_nh3.setMixingRule(2)

feed_nh3 = Stream("H2/N2 Feed", fluid_nh3)
feed_nh3.setPressure(300.0, "bara")
feed_nh3.setTemperature(450.0, "K")
feed_nh3.run()

reactor_nh3 = GibbsReactor("Haber-Bosch", feed_nh3)
reactor_nh3.setEnergyMode(GibbsReactor.EnergyMode.ISOTHERMAL)
reactor_nh3.setDampingComposition(0.05)
reactor_nh3.setMaxIterations(2500)
reactor_nh3.setConvergenceTolerance(1e-6)
reactor_nh3.run()

out_nh3 = reactor_nh3.getOutletStream().getThermoSystem()
z_h2 = out_nh3.getComponent("hydrogen").getz()
z_n2 = out_nh3.getComponent("nitrogen").getz()
z_nh3 = out_nh3.getComponent("ammonia").getz()

print(f"Converged: {reactor_nh3.hasConverged()}")
print(f"Iterations: {reactor_nh3.getActualIterations()}")
print(f"Mass balance error: {reactor_nh3.getMassBalanceError():.6f}%")
print()
print(f"H2:  {z_h2:.4f} ({z_h2*100:.1f}%)")
print(f"N2:  {z_n2:.4f} ({z_n2*100:.1f}%)")
print(f"NH3: {z_nh3:.4f} ({z_nh3*100:.1f}%)")
print(f"\nHeat duty: {reactor_nh3.getPower('kW'):.2f} kW")

Converged: True
Iterations: 263
Mass balance error: 0.001411%

H2:  0.0557 (5.6%)
N2:  0.0186 (1.9%)
NH3: 0.9257 (92.6%)

Heat duty: -0.00 kW


---
## Example 3 — Sulfur Formation from Sour Gas

Simulate Claus-like chemistry: H₂S + O₂ → H₂O + S₈ in a methane-rich background.
Uses CPA equation of state for polar species.

In [5]:
# Cell 5 — Sulfur formation
fluid_s = SystemSrkCPAstatoil(298.15, 1.0)
fluid_s.addComponent("methane", 1e6)
fluid_s.addComponent("H2S", 10.0)
fluid_s.addComponent("oxygen", 2.0)
fluid_s.addComponent("SO2", 0.0)
fluid_s.addComponent("SO3", 0.0)
fluid_s.addComponent("sulfuric acid", 0.0)
fluid_s.addComponent("water", 0.0)
fluid_s.addComponent("S8", 0.0)
fluid_s.setMixingRule(2)

feed_s = Stream("Sour Gas", fluid_s)
feed_s.setPressure(10.0, "bara")
feed_s.setTemperature(100.0, "C")
feed_s.run()

reactor_s = GibbsReactor("Sulfur Reactor", feed_s)
reactor_s.setUseAllDatabaseSpecies(False)
reactor_s.setDampingComposition(0.001)
reactor_s.setMaxIterations(10000)
reactor_s.setConvergenceTolerance(1e-3)
reactor_s.setEnergyMode(GibbsReactor.EnergyMode.ISOTHERMAL)
reactor_s.run()

out_s = reactor_s.getOutletStream().getThermoSystem()
print(f"Converged: {reactor_s.hasConverged()}")
print(f"Iterations: {reactor_s.getActualIterations()}")
print(f"Mass balance converged: {reactor_s.getMassBalanceConverged()}")
print()
print("Outlet composition (ppm):")
for i in range(out_s.getNumberOfComponents()):
    ppm = out_s.getComponent(i).getz() * 1e6
    name = out_s.getComponent(i).getComponentName()
    if ppm > 1e-6:
        print(f"  {name:15s}: {ppm:12.4f} ppm")

Converged: True
Iterations: 1898
Mass balance converged: True

Outlet composition (ppm):
  methane        :  999989.5000 ppm
  H2S            :       5.9999 ppm
  SO2            :       0.0001 ppm
  water          :       4.0000 ppm
  S8             :       0.5000 ppm


---
## Example 4 — Acid Gas Equilibrium in CO₂ Transport

`GibbsReactorCO2` automatically selects the right reaction pathway based on inlet composition.
Here we model a CO₂ pipeline stream with H₂S and O₂ impurities.

In [6]:
# Cell 6 — GibbsReactorCO2: acid gas equilibrium
sys_co2 = SystemSrkEos(275.15, 100.0)
sys_co2.addComponent("CO2", 1e6, "mole/sec")

# Add all potential product species at 0.0
for comp in ["hydrogen", "N2O3", "N2O", "nitrogen", "N2H4", "COS", "ammonia",
             "SO2", "SO3", "NO2", "NO", "water", "H2S", "oxygen",
             "sulfuric acid", "nitric acid", "NH4NO3", "NH4HSO4",
             "formic acid", "acetic acid", "methanol", "ethanol",
             "CO", "NH2OH", "S8", "HNO2"]:
    try:
        sys_co2.addComponent(comp, 0.0, "mole/sec")
    except Exception:
        pass

# Scenario: H2S + O2 + H2O impurities
sys_co2.addComponent("water", 50.0)
sys_co2.addComponent("H2S", 10.0)
sys_co2.addComponent("oxygen", 30.0)
sys_co2.setMixingRule(2)

inlet_co2 = Stream("CO2 Pipeline", sys_co2)
inlet_co2.run()

reactor_co2 = GibbsReactorCO2("Acid Gas Equilibrium", inlet_co2)
reactor_co2.run()

out_co2 = reactor_co2.getOutletStream().getThermoSystem()
print("GibbsReactorCO2 — Acid Gas Equilibrium Results")
print("=" * 50)
print(f"T = {out_co2.getTemperature() - 273.15:.1f} °C, P = {out_co2.getPressure():.1f} bara")
print()
print("Component         Inlet (ppm)  Outlet (ppm)")
print("-" * 45)
# Show key species
for name in ["H2S", "oxygen", "water", "SO2", "SO3", "sulfuric acid", "S8", "NO2", "HNO2"]:
    try:
        ppm_out = out_co2.getComponent(name).getz() * 1e6
        if ppm_out > 1e-6:
            print(f"  {name:15s}             {ppm_out:10.3f}")
    except Exception:
        pass

GibbsReactorCO2 — Acid Gas Equilibrium Results
T = 2.0 °C, P = 100.0 bara

Component         Inlet (ppm)  Outlet (ppm)
---------------------------------------------
  oxygen                          23.213
  water                           59.698
  S8                               1.250
  NO2                              1.565
  HNO2                             0.446


---
## Example 5 — Convergence Diagnostics and Jacobian Inspection

The `GibbsReactor` exposes its Jacobian matrix and objective function for debugging.
This is useful when convergence is slow or fails.

In [7]:
# Cell 7 — Diagnostics: Jacobian and objective function
import numpy as np

# Reuse the ammonia reactor from Example 2
print("=== Convergence Diagnostics ===")
print(f"Converged: {reactor_nh3.hasConverged()}")
print(f"Iterations: {reactor_nh3.getActualIterations()}")
print(f"Final convergence error: {reactor_nh3.getFinalConvergenceError():.2e}")
print(f"Mass balance error: {reactor_nh3.getMassBalanceError():.6f}%")
print()

# Jacobian matrix
J = reactor_nh3.getJacobianMatrix()
if J is not None:
    J_np = np.array([[float(J[i][j]) for j in range(len(J[0]))] for i in range(len(J))])
    row_labels = list(reactor_nh3.getJacobianRowLabels())
    col_labels = list(reactor_nh3.getJacobianColLabels())

    print("Jacobian matrix:")
    header = "              " + "  ".join(f"{c:>12s}" for c in col_labels)
    print(header)
    for i, label in enumerate(row_labels):
        vals = "  ".join(f"{J_np[i, j]:12.4e}" for j in range(J_np.shape[1]))
        print(f"{label:14s}{vals}")

    print(f"\nCondition number: {np.linalg.cond(J_np):.2e}")
    print(f"Jacobian inverse valid: {reactor_nh3.verifyJacobianInverse()}")
else:
    print("Jacobian not available (no variable components)")

# Objective function values
print()
obj_vals = reactor_nh3.getObjectiveFunctionValues()
if obj_vals is not None:
    print("Objective function values (should be ~0 at equilibrium):")
    for key in obj_vals:
        print(f"  {str(key):15s}: {float(obj_vals[key]):.6e} kJ/mol")

=== Convergence Diagnostics ===
Converged: True
Iterations: 263
Final convergence error: 9.58e-07
Mass balance error: 0.001411%

Jacobian matrix:
                n_hydrogen    n_nitrogen     n_ammonia      lambda_N      lambda_H
F_hydrogen      4.8424e+01   -6.9357e+00   -3.3326e+00   -0.0000e+00   -2.0000e+00
F_nitrogen     -6.9357e+00    1.7799e+02   -3.3357e+00   -2.0000e+00   -0.0000e+00
F_ammonia      -3.3326e+00   -3.3357e+00    2.0834e-01   -1.0000e+00   -3.0000e+00
Balance_N       0.0000e+00    2.0000e+00    1.0000e+00    0.0000e+00    0.0000e+00
Balance_H       2.0000e+00    0.0000e+00    3.0000e+00    0.0000e+00    0.0000e+00

Condition number: 6.69e+03
Jacobian inverse valid: True

Objective function values (should be ~0 at equilibrium):
  ammonia        : -2.131249e-06 kJ/mol
  nitrogen       : 4.788616e-05 kJ/mol
  hydrogen       : 4.782064e-05 kJ/mol


---
## Example 6 — Why GibbsReactorCO2 Uses a Two-Stage Approach

Acid gas systems (H₂S + O₂ in dense CO₂) are **too stiff** for a single reactor.
`GibbsReactorCO2` splits the problem into two sequential stages and enables the
consistent Jacobian formulation (`setUseConsistentOffDiagonal(true)`) on each stage.
Below we compare a single-reactor attempt (which does not converge) vs the automatic
two-stage path (which succeeds).

In [9]:
# Cell 8 — Single reactor vs GibbsReactorCO2 two-stage
def make_acid_gas_fluid():
    """Create a standard acid gas test fluid."""
    fluid = SystemSrkEos(275.15, 100.0)
    fluid.addComponent("CO2", 1e6)
    for comp in ["hydrogen", "N2O3", "N2O", "nitrogen", "N2H4", "COS", "ammonia",
                 "SO2", "SO3", "NO2", "NO", "water", "H2S", "oxygen",
                 "sulfuric acid", "nitric acid", "NH4NO3", "NH4HSO4",
                 "formic acid", "acetic acid", "methanol", "ethanol",
                 "CO", "NH2OH", "S8", "HNO2"]:
        try:
            fluid.addComponent(comp, 0.0)
        except Exception:
            pass
    fluid.addComponent("water", 50.0)
    fluid.addComponent("H2S", 10.0)
    fluid.addComponent("oxygen", 30.0)
    fluid.setMixingRule(2)
    return fluid

# Attempt 1: Single GibbsReactor (expected: does not converge)
fluid_single = make_acid_gas_fluid()
feed_single = Stream("Single Feed", fluid_single)
feed_single.run()

r_single = GibbsReactor("Single Reactor", feed_single)
r_single.setUseAllDatabaseSpecies(False)
r_single.setDampingComposition(0.03)
r_single.setMaxIterations(15000)
r_single.setConvergenceTolerance(1e-3)
r_single.setEnergyMode(GibbsReactor.EnergyMode.ISOTHERMAL)
for inert in ["CO", "COS", "CO2", "ammonia", "hydrogen", "N2O3", "nitrogen", "N2H4", "N2O"]:
    r_single.setComponentAsInert(inert)
r_single.run()

# Attempt 2: GibbsReactorCO2 two-stage (expected: converges)
fluid_twostage = make_acid_gas_fluid()
feed_twostage = Stream("Two-Stage Feed", fluid_twostage)
feed_twostage.run()

r_twostage = GibbsReactorCO2("Two-Stage Reactor", feed_twostage)
r_twostage.run()

out_twostage = r_twostage.getOutletStream().getThermoSystem()

print(f"{'Approach':25s} {'Converged':>12s} {'Notes':>30s}")
print("-" * 70)
print(f"{'Single GibbsReactor':25s} {str(bool(r_single.hasConverged())):>12s} {'Too stiff for single stage':>30s}")

outlet_ok = out_twostage is not None
print(f"{'GibbsReactorCO2 (auto)':25s} {str(outlet_ok):>12s} {'Two-stage + consistent J':>30s}")

if outlet_ok:
    print("\nGibbsReactorCO2 outlet (key species, ppm):")
    for name in ["H2S", "oxygen", "water", "SO2", "S8", "sulfuric acid"]:
        try:
            ppm = out_twostage.getComponent(name).getz() * 1e6
            if ppm > 1e-6:
                print(f"  {name:15s}: {ppm:10.3f}")
        except Exception:
            pass

Approach                     Converged                          Notes
----------------------------------------------------------------------
Single GibbsReactor              False     Too stiff for single stage
GibbsReactorCO2 (auto)            True       Two-stage + consistent J

GibbsReactorCO2 outlet (key species, ppm):
  oxygen         :     23.213
  water          :     59.698
  S8             :      1.250


---
## Summary

| Feature | `GibbsReactor` | `GibbsReactorCO2` |
|---------|---------------|-------------------|
| Configuration | Manual (energy mode, damping, inerts) | Automatic (auto-routes based on composition) |
| Energy mode | Isothermal or Adiabatic | Isothermal only |
| Use case | General chemical equilibrium | CO₂ transport acid gas reactions |
| Jacobian diagnostics | Full access (matrix, inverse, labels) | Not exposed (internal reactors) |
| Consistent formulation | Opt-in via `setUseConsistentOffDiagonal(true)` | Auto-enabled for two-stage pathway |

**References:**
- White, Johnson, Dantzig (1958) — RAND method for Gibbs minimisation
- Smith & Missen (1982) — Chemical Reaction Equilibrium Analysis
- Gordon & McBride (1994) — NASA CEA log-mole formulation
- See `docs/process/gibbs-reactor-documentation.md` for the full mathematical derivation